In [1]:
from automol import smiles
from learn_vrc_tst import geom, View
import numpy as np
from scipy.spatial.transform import Rotation

In [2]:
# methane
mol = smiles.geometry("C", opt=True)
geom.translate(mol, -mol.coordinates[0], in_place=True)

vec = mol.coordinates[1]
x, y, z = vec / np.linalg.norm(vec)
theta = np.arccos(z)
phi = np.arctan2(y, x)
rot = Rotation.from_euler('zy', [-phi, -theta + np.pi/4])
mol = geom.rotate(mol, rot, in_place=True)

geom.write_xyz_file(mol, "methane.xyz")

view = View()
view.add_geometry(mol, label=True)
view.add_arrow([5, 0, 0], color="red")
view.add_arrow([0, 1, 0], color="green")
view.add_arrow([0, 0, 1], color="blue")
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [3]:
# OH + OH
# Prepare fragment 1
frag1 = smiles.geometry("[OH]")
cm1 = geom.center_of_mass(frag1)
geom.translate(frag1, -cm1, in_place=True)
geom.rotate(frag1, Rotation.from_euler("z", 90, degrees=True), in_place=True)

# Prepare fragment 2
frag2 = frag1.model_copy(deep=True)

# Separate fragments
geom.translate(frag1, [-1, 0, 0], in_place=True)
geom.translate(frag2, [+1, 0, 0], in_place=True)

geo = geom.concat([frag1, frag2])
cm = geom.center_of_mass(geo)
geom.translate(geo, -cm, in_place=True)

geom.write_xyz_file(geo, "oh+oh.xyz")

view = View()
view.add_geometry(frag1)
view.add_geometry(frag2)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [27]:
from importlib import reload
import pint
reload(geom)

# ethane
# geo = smiles.geometry("C[CH2]", opt=True)
# geo = old_geo.model_copy(deep=True)
geo = geom.read_xyz_file("ethyl_opt.xyz")
geo.coordinates *= pint.Quantity("bohr").m_as("angstrom")

# # set the torsion angle to exactly half
# tors1_keys = [2, 0, 1, 5]
# tors2_keys = [2, 0, 1, 6]
# tors1_angle = geom.dihedral_angle(geo, tors1_keys, degrees=True)
# tors2_angle = geom.dihedral_angle(geo, tors2_keys, degrees=True)
# rot_angle = (tors1_angle + tors2_angle) / 2
# rot_axis = geo.coordinates[1] - geo.coordinates[0]
# rot_axis /= np.linalg.norm(rot_axis)
# geom.rotate(geo, Rotation.from_rotvec(-rot_angle * rot_axis, degrees=True), keys=[1, 5, 6], in_place=True)

# translate to center of mass
cm = geom.center_of_mass(geo)
geom.translate(geo, -cm, in_place=True)

# rotate to inertial frame
rot = geom.rotation_to_inertial_frame(geo)
geom.rotate(geo, rot, in_place=True)

rot_axis = geo.coordinates[1] - geo.coordinates[0]
rot_axis /= np.linalg.norm(rot_axis)
geom.rotate(geo, Rotation.from_rotvec(-170 * rot_axis, degrees=True), in_place=True)

geom.rotate(geo, Rotation.from_euler("y", +3* np.pi/4), in_place=True)

geom.write_xyz_file(geo, "ethyl_opt.xyz")

view = View()
view.add_geometry(geo, label=True)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [40]:
import numpy as np
reload(geom)

geo = geom.read_xyz_file("ethyl_bent.xyz")

trans1 = np.array([0, geo.coordinates[0, 1], 0])
geom.translate(geo, -trans1, in_place=True, keys=[0, 2, 3, 4])
trans2 = np.array([0, geo.coordinates[1, 1], 0])
geom.translate(geo, -trans2, in_place=True, keys=[1, 5, 6])
print(geo.coordinates)

geom.write_xyz_file(geo, "ethyl_bent2.xyz")

geom.view(geo, label=True)

[[-0.481867  0.       -0.338671]
 [ 0.673124  0.        0.621723]
 [-0.666135  1.036502 -0.717506]
 [-0.322923 -0.610513 -1.26319 ]
 [-1.387995 -0.289009  0.202808]
 [ 0.640666  0.649033  1.50143 ]
 [ 1.54513   0.312107 -0.006593]]


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [48]:
import numpy as np
reload(geom)

methane = geom.read_xyz_file("methane_v0.xyz")
oh_oh = geom.read_xyz_file("oh+oh_v0.xyz")
# ethane = geom.read_xyz_file("ethane_v0.xyz")
ethyl = geom.read_xyz_file("ethyl_bent2.xyz")

ax1 = ethyl.coordinates[1] - ethyl.coordinates[0]
ax2 = ethyl.coordinates[2] - ethyl.coordinates[1]
ax3 = np.cross(ax1, ax2)
ax4 = np.cross(ax1, ax3)
ethyl2 = geom.reflect(ethyl, normal=ax4, keys=[1, 5, 6])
# rot_axis = geo.coordinates[1] - geo.coordinates[0]
# rot_axis /= np.linalg.norm(rot_axis)
# ethyl2 = geom.rotate(ethyl, Rotation.from_rotvec(-np.pi * rot_axis), keys=[1, 5, 6])


# geom.translate(oh_oh, [4, 0, 0], in_place=True)
geom.translate(ethyl, [4, 0, 0], in_place=True)
geom.translate(ethyl2, [8, 0, 0], in_place=True)

# geo = geom.concat([methane, oh_oh, ethyl])
geo = geom.concat([methane, ethyl, ethyl2])

geom.write_xyz_file(geo, "symm.xyz")

view = View()
view.add_geometry(geo)
view.add_arrow(ax4 + ethyl.coordinates[0], start_coord=ethyl.coordinates[0])
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.